# A1 · Reducción raw (esorex)

**Spec:** [`docs/spec_A1_codex_raw_reduction.md`](../docs/spec_A1_codex_raw_reduction.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Reduce los raw MUSE con esorex y alinea las exposiciones hasta `cube_telcorr.fits`.

| | |
|---|---|
| **Entrada** | Raw MUSE + calibraciones |
| **Salida (QC/productos)** | `cube_telcorr.fits`, `stages/stage00r_qc.json` |
| **Consume aguas abajo** | Todo el bloque B |


## Cómo ejecutar de forma independiente

> ⚠️ **Etapa no re-ejecutable desde raw en este repo.** En la poda WP-10 se borraron los intermedios regenerables (`muse_scibasic`, `muse_scipost`, …). Se conservaron los productos finales y todo el QC. Este notebook **audita** el producto/QC existente y documenta el comando histórico.

Comando histórico (referencia, requiere los raw + `esorex`):

```bash
conda activate MUSE
bash scripts/reduce_raw.sh
```


## Coste de ejecución (esorex)

> ⏱️ **Referencia real** medida en esta máquina (esorex 3.13.10 / MUSE 2.10.16, dataset NFM-AO de ROXs 12: **7 exposiciones × 24 IFUs = 168 pixtables**).

| Receta | Tiempo | Escala con |
|---|---:|---|
| bias | 33 min | calibración (~fijo) |
| flat | 52 min | calibración |
| wavecal | 51 min | calibración |
| lsf | 50 min | calibración |
| scibasic (std) | 4.5 min | 1× |
| standard | 1.6 min | 1× |
| scibasic (object) | 24 min | **N_exp** |
| scipost | 73 min | **N_exp** |
| **Total (7 exp)** | **≈ 289 min (~4.8 h)** | |

**Fórmula para datos nuevos** (mismo instrumento/máquina):

```
T(min) ≈ T_cal + T_std + N_exp·(t_scibasic + t_scipost) + T_combine
```

con constantes medidas aquí:

- `T_cal ≈ 186 min` = bias+flat+wavecal+lsf. **Una vez por noche/modo**; `0` si reutilizas los master calibrations.
- `T_std ≈ 6 min` = scibasic_std + standard (una vez).
- `t_scibasic ≈ 3.4 min/exp`, `t_scipost ≈ 10.5 min/exp` (24 IFUs c/u; plan B = scipost por exposición).
- `T_combine ≈ 5 min` = muse_exp_align + muse_exp_combine (plan B, offsets manuales).

**Nota:** scibasic/scipost paralelizan sobre los 24 IFUs (OpenMP) → el tiempo escala aprox. inverso al nº de núcleos; `cores_factor` ajusta ese factor respecto a esta máquina base (=1.0). La calibración domina: reutilizar masters recorta ~3 h.


In [ ]:
def estimate_esorex_runtime(n_exp, reuse_calibrations=False, cores_factor=1.0):
    """Estima el wall-time de la reducción esorex (min), calibrada en la
    máquina de referencia (7 exp NFM-AO ~= 289 min). Ver tabla de arriba."""
    T_cal = 0.0 if reuse_calibrations else 186.0  # bias+flat+wavecal+lsf
    T_std = 6.0                                    # scibasic_std + standard
    t_scibasic, t_scipost = 3.4, 10.5             # min por exposición (24 IFU)
    T_combine = 5.0                                # exp_align + exp_combine
    return (T_cal + T_std + n_exp * (t_scibasic + t_scipost) + T_combine) / cores_factor

for n in (1, 3, 7, 10):
    m = estimate_esorex_runtime(n)
    print(f'{n:2d} exp  ->  {m:5.0f} min  (~{m/60:.1f} h)')
print('7 exp reutilizando masters ->',
      f'{estimate_esorex_runtime(7, reuse_calibrations=True):.0f} min')


In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('_nbcommon.py')))
import _nbcommon as nb
RUN_ID = nb.resolve_run_id(None)
print('run =', RUN_ID)
print('dir =', nb.run_dir(RUN_ID))


## Auditar

Etapa de solo-auditoría: se carga el producto/QC más abajo.


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage00r_qc.json', RUN_ID)
nb.show(qc, keys=['shape', 'sha', 'offset', 'esorex', 'muse', 'V1', 'V3', 'V4'], title='A1')


## Decisiones y notas
- **Alineación por plan B (OFFSET_LIST manual)**, no `exp_align` — era espurio; el cubo realineado ≡ ADP a través de B.
- Provenance QC en **AMARILLO**: V1/V3/V4 pass; V2/V5/V6 no disponibles (documentado, no fallos).
- Nada es paper-válido hasta cerrar el A-block (directiva del usuario 2026-07-07).


## Checks


In [ ]:
print('cube shape / provenance:')
nb.show(qc, keys=['shape','offset','esorex'])
